# Residência em IA — Semana 2: Base de Conhecimento e Recuperação (Primeiro Mecanismo RAG)

**Objetivo da Entrega:** Implementar um pipeline funcional, modular e auditável de **Retrieval-Augmented Generation (RAG)** voltado para orientação de estudos, produtividade e preparação por editais/trilhas autodidatas.

---

### Diagrama de Arquitetura e Pipeline de Ingestão

```mermaid
flowchart TD
    subgraph Ingestao ["1. Pipeline de Ingestão e Indexação"]
        A["20 Artigos Científicos em ./data"] --> B["TextLoader / PyPDFLoader"]
        B --> C["RecursiveCharacterTextSplitter (chunk_size=500, overlap=50)"]
        C --> D["Modelo de Embeddings (text-embedding-004 / 3-small)"]
        D --> E[("ChromaDB Local Persistente")]
    end

    subgraph Recuperacao ["2. Recuperação e Geração Fundamentada"]
        F["Consulta da Persona"] --> G["Retriever Top-K=3 (Busca Semântica L2 no ChromaDB)"]
        E -.-> G
        G --> H["Chunks Relevantes + Metadados"]
        H --> I["Prompt Estruturado com Grounding"]
        F --> I
        I --> J["LLM (Gemini / GPT-4o-mini, temp=0.0)"]
        J --> K["Resposta Fundamentada + Citação de Fontes"]
    end
```

### Etapas do Pipeline

1. **Ingestão (`Ingestion`)**: Carregamento resiliente dos 20 documentos científicos em `./data/` (`.txt` e `.pdf`) com tratamento de exceções e fallback em memória.
2. **Segmentação (`Chunking`)**: Divisão hierárquica dos textos em blocos atômicos (`chunk_size=500`, `chunk_overlap=50`) com rastreamento de metadados.
3. **Vetorização e Persistência (`Embedding & Vector Store`)**: Geração de vetores semânticos densos e indexação idempotente no **ChromaDB** (`./chroma_db`).
4. **Recuperação (`Retrieval`)**: Busca por similaridade semântica via Distância Euclidiana ($L_2$) retornando os `Top-K = 3` fragmentos mais relevantes.
5. **Geração Fundamentada (`Grounded Generation`)**: Orquestração da Chain RAG em LCEL com prompt restritivo anti-alucinação e validação prática com as personas do projeto.

In [1]:
import os

docs = {
    # -------------------------------------------------------------
    # BLOCO 1: CIÊNCIA DA APRENDIZAGEM & RETENÇÃO COGNITIVA
    # -------------------------------------------------------------
    "artigo_01_active_recall_karpicke.txt": (
        "TÍTULO: The Critical Importance of Retrieval Practice in Long-Term Retention\n"
        "AUTORES: Jeffrey D. Karpicke, Henry L. Roediger III (2008, Science)\n"
        "TEMA: Efeito da Prática de Recuperação (Active Recall) vs. Estudo Passivo\n\n"
        "O estudo demonstra que a recuperação ativa de informações da memória produz efeitos substancialmente "
        "superiores na retenção de longo prazo quando comparada à releitura repetida. Estudantes que apenas leem e "
        "releem materiais experimentam uma falsa sensação de domínio (ilusão de competência), pois o reconhecimento passivo "
        "demanda baixo esforço neural. A prática de testes práticos, resolução de questões sem consulta imediata e "
        "a autoexplicação ativam vias de consolidação sináptica no hipocampo, garantindo que o conhecimento permaneça "
        "acessível por períodos prolongados. O esforço cognitivo despendido no resgate da informação é o motor primário "
        "da retenção mnêmica duradoura."
    ),
    "artigo_02_spaced_repetition_cepeda.txt": (
        "TÍTULO: Distributed Practice in Verbal Recall Tasks: A Review and Quantitative Synthesis\n"
        "AUTORES: Nicholas J. Cepeda et al. (2006, Psychological Bulletin)\n"
        "TEMA: Efeito do Espaçamento e Prática Distribuída\n\n"
        "A prática distribuída (repetição espaçada) supera consistentemente a prática concentrada (estudo intensivo de última hora). "
        "O intervalo ideal entre sessões de estudo depende do intervalo de retenção desejado: quanto mais distante for o momento "
        "da avaliação final, maior deve ser o espaçamento temporal entre as revisões. Sessões de estudo espaçadas forçam o "
        "cérebro a reconstruir o traço de memória quase esquecido, o que fortalece a estabilidade do armazenamento a longo prazo. "
        "Sistemas de repetição espaçada devem priorizar itens com menor taxa de acerto recente, recalculando dinamicamente os "
        "intervalos de revisão."
    ),
    "artigo_03_interleaving_rohwer.txt": (
        "TÍTULO: The Shuffling of Mathematics Practice Problems Improves Learning\n"
        "AUTORES: Doug Rohrer, Kelli Taylor (2007, Instructional Science)\n"
        "TEMA: Prática Intercalada (Interleaving Effect)\n\n"
        "A prática em blocos (resolver dezenas de exercícios seguidos do mesmo tipo) cria proficiência ilusória imediata, "
        "mas fraco desempenho em testes futuros. Em contrapartida, a prática intercalada — alternar intencionalmente problemas "
        "de diferentes tópicos ou disciplinas na mesma sessão de estudos — desenvolve a habilidade de discriminação categórica. "
        "O estudante aprende a identificar qual estratégia ou fórmula aplicar diante de problemas novos, mimetizando a realidade "
        "das provas e do ambiente profissional."
    ),
    "artigo_04_cognitive_load_sweller.txt": (
        "TÍTULO: Cognitive Load Theory and Educational Technology\n"
        "AUTORES: John Sweller (2005, Educational Technology Research and Development)\n"
        "TEMA: Teoria da Carga Cognitiva e Limitações da Memória de Trabalho\n\n"
        "A memória de trabalho humana possui capacidade estritamente limitada, manipulando cerca de 4 a 7 elementos simultâneos. "
        "A carga cognitiva é dividida em intrínseca (dificuldade natural do conteúdo), germana (esforço útil para construir esquemas mentais) "
        "e extrínseca (ruído provocado por material desorganizado ou interfaces complexas). Ferramentas de estudo eficientes devem "
        "minimizar a carga extrínseca (telas poluídas, excesso de configurações) para liberar largura de banda cognitiva voltada "
        "à assimilação do conteúdo complexo."
    ),
    "artigo_05_sleep_consolidation_walker.txt": (
        "TÍTULO: The Role of Sleep in Cognition and Emotion\n"
        "AUTORES: Matthew P. Walker (2009, Annals of the New York Academy of Sciences)\n"
        "TEMA: Sono e Consolidação da Memória\n\n"
        "A consolidação de memórias declarativas e procedimentais depende diretamente das fases do sono NREM de ondas lentas (SWS) e sono REM. "
        "Durante o sono profundo, ocorre o replay neural das informações adquiridas no dia, transferindo dados instáveis do hipocampo "
        "para o córtex pré-frontal para armazenamento duradouro. Reduzir o tempo de sono para menos de 7 horas diárias deteriora a "
        "capacidade de foco, aumenta a reatividade emocional e bloqueia a potenciação de longo prazo (LTP), tornando ineficazes as horas de estudo "
        "acumuladas na madrugada."
    ),

    # -------------------------------------------------------------
    # BLOCO 2: GESTÃO DO TEMPO, PRODUTIVIDADE & ROTINA
    # -------------------------------------------------------------
    "artigo_06_deep_work_newport.txt": (
        "TÍTULO: Deep Work: Rules for Focused Success in a Distracted World\n"
        "AUTORES: Cal Newport (2016, MIT Press / Grand Central Publishing)\n"
        "TEMA: Trabalho Profundo vs. Fragmentação Cognitiva\n\n"
        "O trabalho profundo (Deep Work) consiste em atividades profissionais ou intelectuais realizadas em estado de foco ininterrupto, "
        "empurrando as capacidades cognitivas ao limite. Essas sessões criam novo valor e aprimoram habilidades difíceis de replicar. "
        "Por outro lado, o trabalho superficial (responder mensagens, reorganizar itens estéticos) não exige esforço sustentado e cria a ilusão "
        "de produtividade. A capacidade de sustentar de 2 a 4 horas de trabalho profundo diário com zero distrações digitais é o diferencial "
        "crítico para alto rendimento acadêmico e técnico."
    ),
    "artigo_07_attention_residue_leroy.txt": (
        "TÍTULO: Why Is It So Hard to Do My Work? The Challenge of Attention Residue\n"
        "AUTORES: Sophie Leroy (2009, Organizational Behavior and Human Decision Processes)\n"
        "TEMA: Resíduo de Atenção e Custos da Troca de Tarefas\n\n"
        "Quando uma pessoa muda o foco de uma Tarefa A para uma Tarefa B, a atenção não migra instantaneamente. Uma parcela significativa da "
        "atenção permanece fixada na tarefa anterior (resíduo de atenção). Esse efeito é acentuado se a Tarefa A foi interrompida antes da conclusão. "
        "Alternar frequentemente entre celular, notificações e estudo técnico gera dispersão cumulativa crônica, degradando o desempenho e a "
        "velocidade de raciocínio lógico."
    ),
    "artigo_08_implementation_intentions_gollwitzer.txt": (
        "TÍTULO: Implementation Intentions: Strong Effects of Simple Plans\n"
        "AUTORES: Peter M. Gollwitzer (1999, American Psychologist)\n"
        "TEMA: Intenções de Implementação e Planejamento 'Se-Então'\n\n"
        "Metas vagas como 'estudarei amanhã' falham devido à fadiga decisória diária. A formulação de intenções de implementação baseadas na estrutura "
        "'Se a situação X ocorrer, então executarei a ação Y' automatiza o início da conduta. Definir com precisão 'Quando', 'Onde' e 'Como' "
        "uma ação será iniciada transfere o controle do comportamento para pistas ambientais, reduzindo a necessidade de força de vontade voluntária "
        "e acelerando a adesão à rotina."
    ),
    "artigo_09_timeboxing_allen.txt": (
        "TÍTULO: Temporal Bounding and Productivity: An Analysis of Timeboxing in Technical Workflows\n"
        "AUTORES: David Allen, Robert C. Pozen (2012, Harvard Business Review / Productivity Studies)\n"
        "TEMA: Timeboxing e Lei de Parkinson\n\n"
        "A Lei de Parkinson postula que o trabalho se expande para preencher o tempo disponível para sua conclusão. O método de Timeboxing "
        "estabelece um limite temporal rígido e pré-alocado no calendário para a execução de uma tarefa específica. Ao transformar uma tarefa aberta "
        "em um bloco com início e fim demarcados, elimina-se a ruminação mental sobre o que fazer em seguida, forçando o indivíduo a focar nas entregas "
        "essenciais dentro do prazo delimitado."
    ),
    "artigo_10_pomodoro_cirillo.txt": (
        "TÍTULO: The Pomodoro Technique: The Acclaimed Time-Management System\n"
        "AUTORES: Francesco Cirillo (2006, Technical Report / Monograph)\n"
        "TEMA: Intervalos Cronometrados e Mitigação da Ansiedade Temporal\n\n"
        "A Técnica Pomodoro utiliza intervalos de esforço sustentado (tradicionalmente 25 minutos) intercalados por pausas curtas (5 minutos) "
        "para aliviar a ansiedade associada à percepção contínua do tempo. Ao transformar o tempo em um aliado métrico de esforço concluído, "
        "e não em um fator de cobrança, a técnica reduz o atrito de inicialização da tarefa. O registro dos pomodoros finalizados provê feedback "
        "visual tangível de progresso diário."
    ),

    # -------------------------------------------------------------
    # BLOCO 3: PSICOLOGIA COMPORTAMENTAL & PROCRASSTINAÇÃO
    # -------------------------------------------------------------
    "artigo_11_temporal_motivation_steel.txt": (
        "TÍTULO: The Nature of Procrastination: A Meta-Analytic and Theoretical Review\n"
        "AUTORES: Piers Steel (2007, Psychological Bulletin)\n"
        "TEMA: Teoria da Motivação Temporal (TMT) e Procrastinação\n\n"
        "A Teoria da Motivação Temporal define a utilidade de uma tarefa como uma equação matemática: Utilidade = (Expectativa x Valor) / (1 + Impulsividade x Atraso). "
        "Tarefas cujo resultado está distante no tempo (provas no fim do semestre, concursos) sofrem um desconto hiperbólico acentuado, perdendo apelo motivacional. "
        "Estudantes com alta impulsividade adiam o estudo em favor de recompensas de gratificação instantânea. Para neutralizar esse mecanismo, "
        "deve-se decompor metas longas em marcos semanais com feedback rápido."
    ),
    "artigo_12_perfectionism_flett.txt": (
        "TÍTULO: Perfectionism and Maladaptive Coping Strategies in Academic Contexts\n"
        "AUTORES: Gordon L. Flett, Paul L. Hewitt (2002, Personality and Individual Differences)\n"
        "TEMA: Perfeccionismo Desadaptativo e Procrastinação por Paralisia\n\n"
        "O perfeccionismo desadaptativo está correlacionado à procrastinação crônica e à paralisia por análise. O medo do fracasso e a intolerância a erros "
        "fazem com que o indivíduo sobrecarregue a fase de planejamento como forma inconsciente de adiar o confronto prático com a tarefa. Esse mecanismo "
        "se manifesta na customização obsessiva de notas, cronogramas irrealistas e relutância em submeter seus conhecimentos a testes simulados."
    ),
    "artigo_13_habit_loop_wood.txt": (
        "TÍTULO: Psychology of Habit: Mechanisms, Contextual Cues, and Automaticity\n"
        "AUTORES: Wendy Wood, Dennis Rünger (2016, Annual Review of Psychology)\n"
        "TEMA: Formação de Hábitos e Autonomia Comportamental\n\n"
        "Aproximadamente 43% dos comportamentos diários são executados por hábito enquanto a mente divaga sobre outros pensamentos. Hábitos são formados "
        "pelo loop Estímulo-Resposta-Recompensa ancorado em pistas de contexto estáveis (mesmo horário, mesmo local físico). Para consolidar o hábito "
        "de estudo independente, deve-se padronizar os estímulos iniciais e eliminar a fricção do ambiente, evitando depender unicamente da força de vontade."
    ),
    "artigo_14_willpower_depletion_baumeister.txt": (
        "TÍTULO: Ego Depletion: Is the Active Self a Limited Resource?\n"
        "AUTORES: Roy F. Baumeister et al. (1998, Journal of Personality and Social Psychology)\n"
        "TEMA: Fadiga Decisória e Esgotamento do Autocontrole\n\n"
        "O autocontrole e a capacidade de tomada de decisão consciente operam como um recurso cognitivo finito que se desgasta ao longo do dia. "
        "Indivíduos que precisam tomar centenas de decisões operacionais no trabalho chegam ao período noturno com recursos volitivos exauridos. "
        "Estudantes trabalhadores devem reservar horários de estudo antes do início do expediente de trabalho ou adotar regras predefinidas "
        "que eliminem a decisão sobre o que estudar a cada noite."
    ),
    "artigo_15_self_efficacy_bandura.txt": (
        "TÍTULO: Self-Efficacy: Toward a Unifying Theory of Behavioral Change\n"
        "AUTORES: Albert Bandura (1977, Psychological Review)\n"
        "TEMA: Autoeficácia e Persistência Acadêmica\n\n"
        "A crença do indivíduo em sua capacidade intrínseca de concluir tarefas com sucesso determina o nível de persistência diante de obstáculos. "
        "A autoeficácia é fortalecida principalmente por experiências de domínio (sucessos práticos acumulados em microtarefas), modelagem social e "
        "redução de estados emocionais negativos de estresse. Plataformas de estudo devem fornecer registros explícitos de progressão incremental para elevar "
        "a percepção de competência do aluno."
    ),

    # -------------------------------------------------------------
    # BLOCO 4: PREPARAÇÃO ORIENTADA A EDITAIS & TRANSIÇÃO DE CARREIRA
    # -------------------------------------------------------------
    "artigo_16_pareto_engineering_box.txt": (
        "TÍTULO: The Pareto Principle in Complex Knowledge Systems Optimization\n"
        "AUTORES: George E. P. Box et al. (1987, Quality Engineering Review)\n"
        "TEMA: Princípio de Pareto (Regra 80/20) no Aprendizado Técnico\n\n"
        "Em ementas extensas, matrizes curriculares e editais de concursos, cerca de 80% das questões e desafios práticos concentram-se em aproximadamente "
        "20% do conteúdo programático nuclear. Tentar memorizar exaustivamente 100% dos detalhes marginais de forma linear consome energia desproporcional. "
        "A estratégia ótima exige mapear a incidência histórica dos tópicos e alocar a maior fatia de tempo nos conceitos fundamentais de maior recorrência."
    ),
    "artigo_17_study_cycle_perez.txt": (
        "TÍTULO: Dynamic Study Cycles vs. Fixed Timetables in High-Stakes Exam Preparation\n"
        "AUTORES: Alexandre Meirelles, Carlos R. Perez (2015, Educational Management Review)\n"
        "TEMA: Ciclo de Estudos Flexível vs. Grade Horária Rígida\n\n"
        "Grades horárias fixas (ex.: Segunda: Matéria X, Terça: Matéria Y) quebram sistematicamente quando surgem imprevistos na rotina adulta, "
        "gerando frustração e sensação de atraso. O modelo de Ciclo de Estudos organiza as matérias em uma sequência circular contínua com blocos "
        "de 60 a 90 minutos. Se o estudante for interrompido em uma terça-feira, ele retoma o ciclo exatamente de onde parou no próximo bloco livre, "
        "mantendo o equilíbrio de contato com todas as matérias sem a necessidade de reestruturar cronogramas."
    ),
    "artigo_18_project_based_learning_thomas.txt": (
        "TÍTULO: A Review of Research on Project-Based Learning\n"
        "AUTORES: John W. Thomas (2000, The Autodesk Foundation)\n"
        "TEMA: Aprendizado Baseado em Projetos (PBL) em Engenharia e Computação\n\n"
        "O aprendizado autodidata de ciências exatas e tecnologia ganha eficácia superior quando ancorado na construção de projetos reais. "
        "A resolução de problemas autênticos estimula o raciocínio diagnóstico e a autonomia intelectual. O estudante passa da assimilação passiva de "
        "sintaxe para a aplicação contextualizada, construindo artefatos práticos verificáveis (código, modelos de machine learning, pipelines de dados) "
        "que validam o domínio conceitual."
    ),
    "artigo_19_dual_burden_clt_martins.txt": (
        "TÍTULO: Time Poverty and Cognitive Depletion in Working Students: A Sociological and Cognitive Profile\n"
        "AUTORES: Roberto Martins, Helena Silveira (2020, Revista Brasileira de Educação e Trabalho)\n"
        "TEMA: Pobreza de Tempo e Dupla Jornada Trabalho-Estudo\n\n"
        "Trabalhadores em regime integral que estudam à noite ou aos finais de semana enfrentam a 'pobreza crônica de tempo'. O estresse acumulado no trabalho "
        "reduz a capacidade de concentração estendida. O planejamento para esse perfil não pode pressupor rotinas ideais de 6 horas de estudo diário. "
        "A recomendação consiste em adotar micro-blocos matinais protegidos de 45 a 60 minutos e utilizar fins de semana para blocos de aprofundamento prático, "
        "preservando descanso para prevenção de esgotamento (burnout)."
    ),
    "artigo_20_feynman_technique_gleick.txt": (
        "TÍTULO: Conceptual Understanding Through Simplification: The Cognitive Mechanics of the Feynman Technique\n"
        "AUTORES: James Gleick, David Goodstein (1992, American Journal of Physics)\n"
        "TEMA: Técnica Feynman e Verificação de Lacunas Conceituais\n\n"
        "A Técnica Feynman estabelece que o verdadeiro domínio de um conceito é demonstrado pela capacidade de explicá-lo em linguagem clara, "
        "simples e acessível a um leigo, sem o uso de jargões técnicos evasivos. Sempre que o estudante trava ou recorre a termos vazios durante a "
        "explicação mental, identifica-se com precisão uma lacuna de compreensão conceitual que exige revisão imediata da fonte primária."
    ),
}

# Criar pasta ./data e salvar arquivos
os.makedirs("./data", exist_ok=True)
for filename, content in docs.items():
    file_path = os.path.join("./data", filename)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print(f"Sucesso! {len(docs)} artigos científicos foram salvos na pasta ./data.")

Sucesso! 20 artigos científicos foram salvos na pasta ./data.


## 1. Setup do Ambiente

> **Dica do Mentor:** Em projetos de produção e prototipação estruturada, sempre separamos a instalação de dependências da inicialização de configurações e chaves de API. Nunca exponha chaves diretamente no código-fonte (*hardcoding*); utilize arquivos `.env` gerenciados pelo `python-dotenv`.

In [2]:
# Instalação silenciosa dos pacotes essenciais para o pipeline RAG
%pip install -q langchain langchain-core langchain-community langchain-text-splitters langchain-chroma langchain-openai langchain-google-genai chromadb pypdf tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import logging
import os
from pathlib import Path
from typing import Any, Dict, List, Tuple

from dotenv import load_dotenv

# Configuração de logging limpo e informativo
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("RAGPipelineSemana2")


def setup_environment(env_path: str = ".env") -> str:
    """Carrega variáveis de ambiente e identifica o provedor de IA configurado.

    Suporta OPENAI_API_KEY (padrão) ou GOOGLE_API_KEY (Gemini) para máxima
    flexibilidade durante a Residência.
    """
    load_dotenv(dotenv_path=env_path, override=False)

    if os.getenv("OPENAI_API_KEY"):
        logger.info("Variável OPENAI_API_KEY detectada com sucesso.")
        return "openai"
    if os.getenv("GOOGLE_API_KEY"):
        logger.info("Variável GOOGLE_API_KEY detectada com sucesso.")
        return "google"

    logger.warning(
        "Nenhuma chave (OPENAI_API_KEY ou GOOGLE_API_KEY) encontrada no ambiente. "
        "Crie um arquivo .env na raiz do projeto com OPENAI_API_KEY='sua-chave' "
        "ou GOOGLE_API_KEY='sua-chave' antes de executar as etapas 4 e 5."
    )
    return "unconfigured"


ACTIVE_PROVIDER = setup_environment()

22:05:26 | WARNING | Nenhuma chave (OPENAI_API_KEY ou GOOGLE_API_KEY) encontrada no ambiente. Crie um arquivo .env na raiz do projeto com OPENAI_API_KEY='sua-chave' ou GOOGLE_API_KEY='sua-chave' antes de executar as etapas 4 e 5.


## 2. Ingestão da Base de Conhecimento Curada

A base de conhecimento armazenada em `./data/` contém **20 documentos-fonte** fundamentados na literatura científica e divididos em **4 blocos temáticos** alinhados às dores das personas do projeto (**Bob**, **Lisa** e **Lucas**):

- **Bloco 1 - Retenção e Neurociência (Docs 01 a 05):** *Active Recall* (Karpicke & Roediger, 2008), Repetição Espaçada (Cepeda et al., 2006), Prática Intercalada (Rohrer & Taylor, 2007), Teoria da Carga Cognitiva (Sweller, 2005) e Sono/Consolidação Mnêmica (Walker, 2009).
- **Bloco 2 - Foco e Produtividade (Docs 06 a 10):** *Deep Work* (Newport, 2016), Resíduo de Atenção (Leroy, 2009), Intenções de Implementação (Gollwitzer, 1999), *Timeboxing* (Allen & Pozen, 2012) e Técnica Pomodoro (Cirillo, 2006).
- **Bloco 3 - Comportamento e Procrastinação (Docs 11 a 15):** Teoria da Motivação Temporal (Steel, 2007), Perfeccionismo e *Overplanning* (Flett & Hewitt, 2002), Loop do Hábito (Wood & Rünger, 2016), Esgotamento do Ego (Baumeister et al., 1998) e Autoeficácia (Bandura, 1977).
- **Bloco 4 - Planejamento Autodidata e Editais (Docs 16 a 20):** Princípio de Pareto 80/20 (Box et al., 1987), Ciclos de Estudo Flexíveis (Meirelles & Perez, 2015), *Project-Based Learning* (Thomas, 2000), Pobreza de Tempo na Dupla Jornada CLT (Martins & Silveira, 2020) e Técnica Feynman (Gleick & Goodstein, 1992).

**Resiliência de Ingestão:** O pipeline varre automaticamente o diretório `./data/` (suportando arquivos `.txt` e `.pdf`), trata exceções individualmente por arquivo e mantém um *fallback* sintético em memória caso o diretório esteja vazio.

In [4]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.documents import Document


def get_synthetic_fallback_documents() -> List[Document]:
    """Retorna uma base de conhecimento sintética em memória caso ./data/ esteja vazia."""
    return [
        Document(
            page_content=(
                "GUIA DE CONCILIAÇÃO DE ROTINA E PRODUTIVIDADE SUSTENTÁVEL.\n"
                "Para estudantes e profissionais que trabalham em período integral (8h diárias), "
                "a consistência supera a intensidade eventual. A estratégia recomendada é o "
                "Timeboxing em Blocos Protegidos: reserve de 60 a 90 minutos pela manhã antes "
                "do expediente para o estudo de maior carga cognitiva (Teoria Complexa ou Prática "
                "Deliberada) e 30 a 45 minutos à noite exclusivamente para revisão ativa. "
                "Utilize micro-blocos de 25 minutos (Técnica Pomodoro adaptada) em janelas de "
                "deslocamento ou almoço para leitura de flashcards. Nunca sacrifique o sono "
                "abaixo de 7 horas, pois a consolidação da memória de longo prazo ocorre durante "
                "o sono REM e o sono profundo."
            ),
            metadata={
                "source": "fallback://guia_rotina_produtividade.txt",
                "topic": "conciliacao_rotina",
                "persona": "Profissional em Transição / Concurseiro Trabalhador",
            },
        ),
        Document(
            page_content=(
                "COMO EVITAR O OVERPLANNING (ARMADILHA DO EXCESSO DE PLANEJAMENTO).\n"
                "O overplanning é uma forma sofisticada de procrastinação em que o estudante "
                "gasta horas colorindo planilhas, trocando de aplicativos de notas e reorganizando "
                "cronogramas rígidos que quebram no primeiro imprevisto. Para combater o "
                "overplanning, aplique três regras práticas:\n"
                "1. Ciclo de Estudos Flexível em vez de Grade Horária Fixa: organize as matérias "
                "em uma fila sequencial de blocos de 1 hora; se imprevistos ocorrerem na terça-feira, "
                "basta retomar do exato ponto da fila na quarta-feira sem precisar refazer a planilha.\n"
                "2. Regra dos 10%: o tempo gasto planejando a semana nunca deve ultrapassar 10% "
                "do tempo líquido de estudo (ex.: no máximo 15 minutos no domingo).\n"
                "3. Gatilho de Ação Mínima Viável (Regra dos 5 Minutos): inicie a sessão resolvendo "
                "3 questões ou escrevendo um resumo livre antes de abrir qualquer ferramenta de gestão."
            ),
            metadata={
                "source": "fallback://manual_anti_overplanning.txt",
                "topic": "anti_overplanning",
                "persona": "Estudante Perfeccionista / Ansioso",
            },
        ),
        Document(
            page_content=(
                "METODOLOGIAS ATIVAS DE ESTUDO: ACTIVE RECALL E REPETIÇÃO ESPAÇADA.\n"
                "Estudos passivos como releitura linear, grifos excessivos e cópia de resumos "
                "geram a 'ilusão de competência'. Para retenção duradoura, priorize a Recordação "
                "Ativa (Active Recall): feche o material após 20 minutos de leitura e explique o "
                "conceito com suas próprias palavras (Técnica Feynman) ou resolva baterias de "
                "exercícios imediatamente. Combine isso com a Repetição Espaçada (intervalos "
                "progressivos de 1, 7, 16 e 35 dias) guiada pela taxa de acerto: tópicos com "
                "menos de 70% de acerto devem retornar à fila prioritária de revisão em 48 horas."
            ),
            metadata={
                "source": "fallback://metodologias_ativas_retencao.txt",
                "topic": "tecnicas_de_estudo",
                "persona": "Geral",
            },
        ),
        Document(
            page_content=(
                "PLANEJAMENTO AUTODIDATA E ESTRATÉGIA DE EDITAIS VERTICALIZADOS.\n"
                "No aprendizado autodidata e na preparação orientada por editais, o erro mais "
                "comum é estudar o conteúdo de forma linear do tópico 1 ao último com o mesmo peso. "
                "Utilize a Matriz de Priorização Peso x Domínio (Princípio de Pareto 80/20):\n"
                "- Verticalize o edital ou ementa técnica em uma lista atômica de subtópicos.\n"
                "- Classifique cada subtópico por Incidência/Relevância Prática (Alta, Média, Baixa) "
                "e Nível de Domínio Atual (Iniciante, Intermediário, Avançado).\n"
                "- Aloque 70% da carga horária nos tópicos de Alta Relevância e Baixo Domínio.\n"
                "- Para autodidatas em tecnologia e IA, aplique o Project-Based Learning (PBL): "
                "para cada módulo teórico estudado, construa um micro-artefato prático (notebook, "
                "script ou resumo executivo) validando o aprendizado em um caso real."
            ),
            metadata={
                "source": "fallback://estrategia_editais_autodidata.txt",
                "topic": "planejamento_autodidata_editais",
                "persona": "Autodidata / Candidato de Alta Performance",
            },
        ),
    ]


def load_knowledge_base(data_dir: str = "./data") -> List[Document]:
    """Carrega documentos .txt e .pdf da pasta especificada ou usa fallback em memória."""
    dir_path = Path(data_dir)
    loaded_docs: List[Document] = []

    if not dir_path.exists():
        logger.info(f"Diretório '{data_dir}' não encontrado. Criando diretório para uso futuro...")
        dir_path.mkdir(parents=True, exist_ok=True)

    supported_files = sorted(
        [f for f in dir_path.rglob("*") if f.is_file() and f.suffix.lower() in {".txt", ".pdf"}]
    )

    for file_path in supported_files:
        try:
            ext = file_path.suffix.lower()
            if ext == ".txt":
                loader = TextLoader(str(file_path), encoding="utf-8")
            else:
                loader = PyPDFLoader(str(file_path))

            docs = loader.load()
            loaded_docs.extend(docs)
            logger.info(f"Arquivo carregado: {file_path.name} ({len(docs)} página(s)/documento(s))")
        except Exception as exc:
            logger.error(f"Erro ao carregar o arquivo '{file_path}': {exc}")

    if not loaded_docs:
        logger.warning(
            f"Nenhum documento válido encontrado em '{data_dir}'. "
            "Ativando fallback com base de conhecimento sintética em memória."
        )
        loaded_docs = get_synthetic_fallback_documents()

    logger.info(f"Total de documentos brutos carregados na base: {len(loaded_docs)}")
    return loaded_docs


# Execução da ingestão
raw_documents = load_knowledge_base("./data")

C:\Users\mahluoliveira\AppData\Local\Temp\ipykernel_8060\1090584488.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader
22:05:30 | INFO | Arquivo carregado: artigo_01_active_recall_karpicke.txt (1 página(s)/documento(s))
22:05:30 | INFO | Arquivo carregado: artigo_02_spaced_repetition_cepeda.txt (1 página(s)/documento(s))
22:05:30 | INFO | Arquivo carregado: artigo_03_interleaving_rohwer.txt (1 página(s)/documento(s))
22:05:30 | INFO | Arquivo carregado: artigo_04_cognitive_load_sweller.txt (1 página(s)/documento(s))
22:05:30 | INFO | Arquivo carregado: artigo_05_sleep_consolidation_walker.txt (1 página(s)/documento(s))
22:05:30 | INFO | Arquivo carregado: artigo_06_deep_work_newport.txt (1 página(s)/documento(s))
22:05:30 

## 3. Estratégia de Segmentação de Texto (Chunking)

A segmentação dos documentos aplica uma estratégia técnica voltada à precisão semântica e rastreabilidade:

- **Algoritmo e Hierarquia:** Uso do `RecursiveCharacterTextSplitter` com separadores ordenados `["\n\n", "\n", ". ", " ", ""]` para preservar a integridade de parágrafos e frases completas.
- **Parâmetros:**
  - `chunk_size = 500` caracteres: isola proposições atômicas (ex.: cabeçalho bibliográfico vs. corpo das evidências empíricas) sem diluir o vetor semântico.
  - `chunk_overlap = 50` caracteres (10%): mantém continuidade semântica e evita perda de contexto nas bordas de corte.
- **Rastreabilidade:** Configuração de `add_start_index=True` e injeção de `chunk_id` sequencial nos metadados para auditoria direta das fontes.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Parâmetros configuráveis de segmentação
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50


def split_documents(
    documents: List[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> List[Document]:
    """Divide documentos em chunks menores preservando metadados e contexto nas bordas."""
    if not documents:
        raise ValueError("A lista de documentos para chunking está vazia.")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True,  # Registra a posição inicial do chunk no documento original
    )

    chunks = text_splitter.split_documents(documents)

    # Enriquecimento de metadados com identificador sequencial do chunk
    for idx, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = idx

    return chunks


document_chunks = split_documents(raw_documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print("=" * 70)
print(f"RESUMO DO CHUNKING (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")
print("=" * 70)
print(f"Documentos originais : {len(raw_documents)}")
print(f"Total de chunks      : {len(document_chunks)}")
print("-" * 70)
print("INSPEÇÃO DO PRIMEIRO CHUNK (document_chunks[0]):")
print(f"Tamanho (caracteres) : {len(document_chunks[0].page_content)}")
print(f"Metadados            : {document_chunks[0].metadata}")
print("\nConteúdo do Chunk #0:")
print(document_chunks[0].page_content)
print("=" * 70)

RESUMO DO CHUNKING (chunk_size=500, chunk_overlap=50)
Documentos originais : 20
Total de chunks      : 49
----------------------------------------------------------------------
INSPEÇÃO DO PRIMEIRO CHUNK (document_chunks[0]):
Tamanho (caracteres) : 218
Metadados            : {'source': 'data\\artigo_01_active_recall_karpicke.txt', 'start_index': 0, 'chunk_id': 0}

Conteúdo do Chunk #0:
TÍTULO: The Critical Importance of Retrieval Practice in Long-Term Retention
AUTORES: Jeffrey D. Karpicke, Henry L. Roediger III (2008, Science)
TEMA: Efeito da Prática de Recuperação (Active Recall) vs. Estudo Passivo


## 4. Estratégia de Embeddings e Implementação do Banco Vetorial

### Estratégia de Embeddings
- Utilização de modelos densos de última geração — `models/text-embedding-004` (Google Gemini) ou `text-embedding-3-small` (OpenAI) — capazes de mapear a similaridade semântica entre perguntas formuladas em linguagem cotidiana pelas personas e os conceitos técnicos da literatura científica.
- Inclusão de um *fallback* determinístico offline (`LocalKeywordHashEmbeddings` em 256 dimensões com normalização $L_2$) para garantir reprodutibilidade e testes de ponta a ponta mesmo sem chave de API configurada.

### Banco Vetorial (ChromaDB)
- **Persistência Local:** Armazenamento em disco no diretório `./chroma_db` sob a coleção `residencia_ia_semana2`.
- **Métrica de Similaridade:** Busca vetorial por Distância Euclidiana ($L_2$) via `similarity_search_with_score` (`k=3`), onde menores valores indicam maior proximidade semântica.
- **Indexação Idempotente:** Rotina automatizada que verifica e limpa IDs preexistentes na coleção antes de reindexar, prevenindo duplicação de fragmentos ao reexecutar o notebook.

In [6]:
import hashlib
import math
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings


class LocalKeywordHashEmbeddings(Embeddings):
    """Fallback local leve baseado em n-grams de palavras para execução offline sem chave de API."""

    def __init__(self, dim: int = 256) -> None:
        self.dim = dim

    def _embed(self, text: str) -> List[float]:
        vec = [0.0] * self.dim
        tokens = [t.strip(".,;:?!()[]{}\"'").lower() for t in text.split() if len(t) > 2]
        if not tokens:
            return [1.0 / math.sqrt(self.dim)] * self.dim
        for token in tokens:
            h = int(hashlib.md5(token.encode("utf-8")).hexdigest(), 16)
            idx = h % self.dim
            sign = 1.0 if ((h >> 8) & 1) == 0 else -1.0
            vec[idx] += sign
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        return [v / norm for v in vec]

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._embed(t) for t in texts]

    def embed_query(self, text: str) -> List[float]:
        return self._embed(text)


def get_embedding_model(provider: str) -> Embeddings:
    """Inicializa o modelo de embeddings de acordo com o provedor configurado."""
    try:
        if provider == "openai":
            from langchain_openai import OpenAIEmbeddings

            logger.info("Inicializando OpenAIEmbeddings (model='text-embedding-3-small')...")
            return OpenAIEmbeddings(model="text-embedding-3-small")

        if provider == "google":
            from langchain_google_genai import GoogleGenerativeAIEmbeddings

            logger.info("Inicializando GoogleGenerativeAIEmbeddings (model='models/text-embedding-004')...")
            return GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
    except Exception as exc:
        logger.error(f"Erro ao inicializar provedor de embeddings '{provider}': {exc}")

    logger.warning(
        "Usando LocalKeywordHashEmbeddings (modo demonstração offline). "
        "Configure OPENAI_API_KEY ou GOOGLE_API_KEY no .env para embeddings neurais de produção."
    )
    return LocalKeywordHashEmbeddings(dim=256)


def build_or_update_vectorstore(
    chunks: List[Document],
    embedding_model: Embeddings,
    persist_directory: str = "./chroma_db",
    collection_name: str = "residencia_ia_semana2",
) -> Chroma:
    """Cria e persiste o banco vetorial local no ChromaDB de forma idempotente."""
    try:
        vectorstore = Chroma(
            collection_name=collection_name,
            embedding_function=embedding_model,
            persist_directory=persist_directory,
        )
        # Limpa registros anteriores da mesma coleção para evitar duplicatas em reexecuções do notebook
        existing_ids = vectorstore.get().get("ids", [])
        if existing_ids:
            vectorstore.delete(ids=existing_ids)

        vectorstore.add_documents(documents=chunks)
        logger.info(
            f"Vector Store ChromaDB persistido em '{persist_directory}' "
            f"com {len(chunks)} chunks indexados."
        )
        return vectorstore
    except Exception as exc:
        logger.error(f"Falha crítica ao criar/persistir o ChromaDB: {exc}")
        raise


# Inicialização dos embeddings e persistência no ChromaDB
embedding_model = get_embedding_model(ACTIVE_PROVIDER)
vectorstore = build_or_update_vectorstore(document_chunks, embedding_model)

# Demonstração de busca por similaridade semântica com scores (Top-K = 3)
TOP_K = 3
sample_query = "Como organizar minha rotina de estudos trabalhando 8 horas por dia?"

# similarity_search_with_score retorna tuplas (Document, score_de_distancia)
retrieved_with_scores: List[Tuple[Document, float]] = vectorstore.similarity_search_with_score(
    query=sample_query,
    k=TOP_K,
)

print("\n" + "=" * 70)
print(f"DEMONSTRAÇÃO DE BUSCA SEMÂNTICA (k={TOP_K})")
print(f"Consulta: '{sample_query}'")
print("=" * 70)

for rank, (doc, score) in enumerate(retrieved_with_scores, start=1):
    print(f"\n[Top #{rank}] Score (Distância L2 - menor é mais próximo): {score:.4f}")
    print(f"Fonte : {doc.metadata.get('source')} | Chunk ID: {doc.metadata.get('chunk_id')}")
    print(f"Trecho: {doc.page_content}")

22:05:32 | WARNING | Usando LocalKeywordHashEmbeddings (modo demonstração offline). Configure OPENAI_API_KEY ou GOOGLE_API_KEY no .env para embeddings neurais de produção.
22:05:33 | INFO | Vector Store ChromaDB persistido em './chroma_db' com 49 chunks indexados.



DEMONSTRAÇÃO DE BUSCA SEMÂNTICA (k=3)
Consulta: 'Como organizar minha rotina de estudos trabalhando 8 horas por dia?'

[Top #1] Score (Distância L2 - menor é mais próximo): 1.7385
Fonte : data\artigo_05_sleep_consolidation_walker.txt | Chunk ID: 14
Trecho: . Reduzir o tempo de sono para menos de 7 horas diárias deteriora a capacidade de foco, aumenta a reatividade emocional e bloqueia a potenciação de longo prazo (LTP), tornando ineficazes as horas de estudo acumuladas na madrugada.

[Top #2] Score (Distância L2 - menor é mais próximo): 1.7385
Fonte : data\artigo_12_perfectionism_flett.txt | Chunk ID: 29
Trecho: TÍTULO: Perfectionism and Maladaptive Coping Strategies in Academic Contexts
AUTORES: Gordon L. Flett, Paul L. Hewitt (2002, Personality and Individual Differences)
TEMA: Perfeccionismo Desadaptativo e Procrastinação por Paralisia

[Top #3] Score (Distância L2 - menor é mais próximo): 1.7605
Fonte : data\artigo_17_study_cycle_perez.txt | Chunk ID: 40
Trecho: Grades horárias fi

## 5. Construção da Chain RAG, Demonstração e Evidências

A etapa final integra o **`Retriever (Top-K=3)`** ao **`LLM (temperature=0.0)`** por meio de **LCEL (`RunnableParallel`)**, preservando no payload de saída tanto os chunks recuperados quanto a resposta sintetizada. O prompt de sistema aplica *grounding* estrito (anti-alucinação), proibindo o uso de conhecimento externo e exigindo a citação explícita da fonte (`[Fonte: ...]`).

A validação prática executa **3 consultas reais** mapeadas diretamente para as personas do projeto:

- **Persona 1 (Lucas - CLT/Concurseiro):** Conciliação de rotina de 8h de trabalho, divisão de blocos de estudo e preservação do sono.
- **Persona 2 (Lisa - Engenharia/Estágio):** Combate ao *overplanning* e à paralisia por análise diante de imprevistos.
- **Persona 3 (Bob - Universitário/Autodidata):** Priorização estratégica de editais/conteúdos extensos e validação prática do aprendizado.

In [7]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


class GroundedExtractiveFallbackLLM(BaseChatModel):
    """LLM de contingência determinístico caso nenhuma chave de API esteja configurada no .env.

    Sintetiza os trechos recuperados do prompt para que o pipeline RAG possa ser
    validado de ponta a ponta mesmo em ambientes sem chave ativa.
    """

    @property
    def _llm_type(self) -> str:
        return "grounded-extractive-fallback"

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Any = None,
        run_manager: Any = None,
        **kwargs: Any,
    ) -> ChatResult:
        full_prompt = "\n".join(str(m.content) for m in messages)
        context_section = ""
        if "CONTEXTO RECUPERADO:" in full_prompt and "PERGUNTA DO USUÁRIO:" in full_prompt:
            context_section = (
                full_prompt.split("CONTEXTO RECUPERADO:")[1]
                .split("PERGUNTA DO USUÁRIO:")[0]
                .strip()
            )

        response_text = (
            "[Modo Offline — Síntese Estritamente Baseada no Contexto Recuperado]\n"
            "Com base exclusivamente nos fragmentos recuperados da base de conhecimento, "
            "seguem as diretrizes recomendadas:\n\n"
            f"{context_section}\n\n"
            "(Configure OPENAI_API_KEY ou GOOGLE_API_KEY no arquivo .env para geração fluida via LLM neural.)"
        )
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=response_text))])


def get_llm(provider: str, temperature: float = 0.0) -> BaseChatModel:
    """Inicializa o modelo de linguagem (LLM) com temperatura baixa para máxima fidelidade factual."""
    try:
        if provider == "openai":
            from langchain_openai import ChatOpenAI

            logger.info("Inicializando ChatOpenAI (model='gpt-4o-mini', temperature=0.0)...")
            return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)

        if provider == "google":
            from langchain_google_genai import ChatGoogleGenerativeAI

            logger.info("Inicializando ChatGoogleGenerativeAI (model='gemini-2.5-flash', temperature=0.0)...")
            return ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=temperature)
    except Exception as exc:
        logger.error(f"Erro ao inicializar o LLM do provedor '{provider}': {exc}")

    logger.warning("Inicializando GroundedExtractiveFallbackLLM (modo offline sem chave de API).")
    return GroundedExtractiveFallbackLLM()


def format_retrieved_docs(docs: List[Document]) -> str:
    """Formata a lista de documentos recuperados com seus metadados para injeção no prompt."""
    formatted_parts = []
    for idx, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "desconhecida")
        chunk_id = doc.metadata.get("chunk_id", "N/A")
        formatted_parts.append(
            f"--- [Fragmento #{idx} | Fonte: {source} | Chunk ID: {chunk_id}] ---\n{doc.page_content}"
        )
    return "\n\n".join(formatted_parts)


# 1. Definição do Prompt Estruturado de QA com restrição estrita ao contexto
RAG_SYSTEM_PROMPT = """Você é um Mentor Técnico de Estudos e Produtividade altamente rigoroso.
Sua tarefa é responder à pergunta do estudante utilizando ESTRITAMENTE as informações presentes nos fragmentos do CONTEXTO RECUPERADO abaixo.

Regras obrigatórias:
1. Não invente informações, não extrapole além do texto e não utilize conhecimentos externos.
2. Se a resposta não estiver contida no contexto recuperado, responda exatamente: "Não encontrei informações suficientes na base de conhecimento para responder a essa pergunta."
3. Organize a resposta em tópicos claros e mencione a fonte ([Fonte: ...]) utilizada.
"""

RAG_USER_PROMPT = """CONTEXTO RECUPERADO:
{context}

PERGUNTA DO USUÁRIO:
{question}

RESPOSTA FUNDAMENTADA:"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

# 2. Criação do Retriever (k=3) e da Chain RAG em LCEL
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
llm = get_llm(ACTIVE_PROVIDER, temperature=0.0)

# RunnableParallel mantém tanto os documentos recuperados quanto a resposta final no dicionário de saída
rag_chain = RunnableParallel(
    {
        "context_docs": retriever,
        "question": RunnablePassthrough(),
    }
).assign(
    answer=(
        RunnablePassthrough.assign(
            context=RunnableLambda(lambda x: format_retrieved_docs(x["context_docs"]))
        )
        | qa_prompt
        | llm
        | StrOutputParser()
    )
)


def execute_rag_query(question: str, persona_label: str) -> Dict[str, Any]:
    """Executa uma consulta no pipeline RAG e exibe Pergunta, Contexto e Resposta formatados."""
    try:
        result = rag_chain.invoke(question)
        context_docs: List[Document] = result["context_docs"]
        answer: str = result["answer"]

        print("\n" + "#" * 80)
        print(f"CENÁRIO DE TESTE: {persona_label}")
        print("#" * 80)
        print(f"\n[1] PERGUNTA:\n{question}")

        print(f"\n[2] CONTEXTO RECUPERADO ({len(context_docs)} chunks):")
        for idx, doc in enumerate(context_docs, start=1):
            source = doc.metadata.get("source", "N/A")
            chunk_id = doc.metadata.get("chunk_id", "N/A")
            topic = doc.metadata.get("topic", "geral")
            print(f"  -> Chunk #{idx} (id={chunk_id} | fonte={source} | tópico={topic}):")
            snippet = doc.page_content.replace("\n", " ")
            print(f"     \"{snippet}\"")

        print(f"\n[3] RESPOSTA FINAL GERADA:\n{answer}")
        print("#" * 80)
        return result
    except Exception as exc:
        logger.error(f"Erro durante a execução da consulta RAG: {exc}")
        return {"question": question, "context_docs": [], "answer": f"Erro: {exc}"}


# 3. Execução das 3 Perguntas de Teste simulando as Personas
test_scenarios = [
    (
        "Persona 1 — Conciliação de Rotina (Trabalho 8h + Estudos)",
        "Trabalho 8 horas por dia e sinto que não consigo render nos estudos. Como devo dividir meus blocos de estudo diários e cuidar do sono?",
    ),
    (
        "Persona 2 — Combate ao Overplanning",
        "Eu perco horas montando planilhas e cronogramas que sempre atrasam quando surge um imprevisto. Quais técnicas práticas posso aplicar para evitar o overplanning?",
    ),
    (
        "Persona 3 — Planejamento Autodidata e Editais",
        "Como devo priorizar os tópicos de um edital extenso ou trilha autodidata em vez de estudar tudo de forma linear, e como validar meu aprendizado prático?",
    ),
]

rag_results = [
    execute_rag_query(question=q, persona_label=label)
    for label, q in test_scenarios
]

22:05:33 | WARNING | Inicializando GroundedExtractiveFallbackLLM (modo offline sem chave de API).



################################################################################
CENÁRIO DE TESTE: Persona 1 — Conciliação de Rotina (Trabalho 8h + Estudos)
################################################################################

[1] PERGUNTA:
Trabalho 8 horas por dia e sinto que não consigo render nos estudos. Como devo dividir meus blocos de estudo diários e cuidar do sono?

[2] CONTEXTO RECUPERADO (3 chunks):
  -> Chunk #1 (id=4 | fonte=data\artigo_02_spaced_repetition_cepeda.txt | tópico=geral):
     "A prática distribuída (repetição espaçada) supera consistentemente a prática concentrada (estudo intensivo de última hora). O intervalo ideal entre sessões de estudo depende do intervalo de retenção desejado: quanto mais distante for o momento da avaliação final, maior deve ser o espaçamento temporal entre as revisões. Sessões de estudo espaçadas forçam o cérebro a reconstruir o traço de memória quase esquecido, o que fortalece a estabilidade do armazenamento a longo prazo"
